In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi

Sat Sep 26 04:21:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/LLMValidation_LabeledIn")

INPUT_FILE = BASE_DIR / "LabeledIn_LLM_Benchmark.xlsx"

RESULT_FILE = BASE_DIR / "LabeledIn_Qwen3_8B_Results.xlsx"

print("BASE_DIR:", BASE_DIR)
print("Benchmark exists:", INPUT_FILE.exists())
print("Result file:", RESULT_FILE)

BASE_DIR: /content/drive/MyDrive/LLMValidation_LabeledIn
Benchmark exists: True
Result file: /content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_Qwen3_8B_Results.xlsx


In [ ]:
!pip -q install pandas openpyxl requests json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.7 MB/s eta 0:00:00


In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 81 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (521 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zs

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama --version

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Ollama server started.")

Ollama server started.


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
)

print("Status code:", response.status_code)
print(response.json())

Status code: 200
{'models': []}


In [ ]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [ ]:
!ollama pull qwen3:8b

In [ ]:
!ollama list

NAME        ID              SIZE      MODIFIED       
qwen3:8b    500a1f067a9f    5.2 GB    35 seconds ago    


In [ ]:
import requests
import json

payload = {
    "model": "qwen3:8b",
    "messages": [
        {
            "role": "user",
            "content": "Return only JSON: {\"diseases\":[\"hypertension\"]}"
        }
    ],
    "stream": False,
    "think": False
}

response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json=payload,
    timeout=240
)

print(response.json()["message"]["content"])

{"diseases":["hypertension"]}


In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/LLMValidation_LabeledIn")

INPUT_FILE = BASE_DIR / "LabeledIn_LLM_Benchmark.xlsx"
RESULT_FILE = BASE_DIR / "LabeledIn_Qwen3_8B_Results.xlsx"

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Benchmark"
)

print("Number of records:", len(df))
print("Columns:")
print(df.columns.tolist())

assert len(df) == 500, (
    f"Expected 500 records but found {len(df)}"
)

print("✓ Full 500-record benchmark loaded successfully.")

Number of records: 500
Columns:
['record_id', 'dailymed_setid', 'indication_text', 'gold_expert_mentions', 'gold_umls_cuis', 'gold_cui_mention_pairs', 'gold_unique_mention_count', 'gold_unique_cui_count', 'annotation_occurrence_count', 'word_count', 'related_setid_count', 'gold_source', 'openbiollm_output', 'openbiollm_status', 'pubmed_output', 'pubmed_status', 'gemma_output', 'gemma_status', 'medgemma_output', 'medgemma_status']
✓ Full 500-record benchmark loaded successfully.


In [ ]:
display(
    df[
        [
            "record_id",
            "indication_text",
            "gold_expert_mentions",
            "gold_umls_cuis"
        ]
    ].head()
)

,record_id,indication_text,gold_expert_mentions,gold_umls_cuis
0,10004,Pantoprazole sodium delayed release tablets ar...,Erosive Esophagitis; Gastroesophageal Reflux D...,C0267055; C0017168; C0043515
1,10008,Anxiety Disorders Alprazolam tablets are indic...,Anxiety Disorders; anxiety disorder; Anxiety; ...,C0003469; C0338908; C0030319; C0001818
2,10012,Amiodarone hydrochloride injection is indicate...,ventricular fibrillation; ventricular tachycar...,C0042510; C0042514
3,10016,"Clobetasol propionate foam, 0.05% is a super p...",dermatoses; plaque type psoriasis,C0037274; C0406317
4,10017,Metaxalone tablets are indicated as an adjunct...,"acute, painful",C0184567


In [ ]:
for column in [
    "qwen3_output",
    "qwen3_raw",
    "qwen3_status",
    "qwen3_runtime_sec"
]:
    if column not in df.columns:
        df[column] = ""

print("Qwen result columns created.")

Qwen result columns created.


In [ ]:
df.to_excel(
    RESULT_FILE,
    index=False
)

print("Initial Qwen result file saved:")
print(RESULT_FILE)

Initial Qwen result file saved:
/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_Qwen3_8B_Results.xlsx


In [ ]:
import json
import time
import requests

MODEL = "qwen3:8b"

URL = "http://127.0.0.1:11434/api/chat"

SYSTEM_MESSAGE = """
You are a biomedical information extraction system.
Follow the user's extraction instructions exactly.
Return only the requested JSON output.
""".strip()

EXTRACTION_TEMPLATE = """
Extract ALL diseases, disorders, syndromes, symptoms, or clinical conditions
for which the drug is explicitly indicated in the following
Indications and Usage text.

Rules:

1. Read the entire supplied text.
2. Include every explicitly stated clinical indication.
3. Include symptoms or clinical conditions when explicitly indicated
   for treatment, prevention, management, or relief.
4. Preserve clinically meaningful disease subtypes and modifiers.
5. Use wording from the supplied text as closely as possible.
6. Do not intentionally normalize or generalize disease names.
7. Do not infer conditions that are not explicitly supported by the text.
8. Exclude conditions explicitly stated as NOT indicated or NOT recommended.
9. Exclude causative organisms when they merely describe the cause of infection.
10. Exclude mechanisms, procedures, dosing information,
    treatment regimens, and patient demographics.
11. Remove duplicate clinical conditions.

Return ONLY valid JSON:

{{"diseases":["condition 1","condition 2"]}}

If there are genuinely no indications:

{{"diseases":[]}}

INDICATIONS AND USAGE:

{indication_text}
""".strip()

In [ ]:
def parse_output(raw_output):

    raw_output = str(raw_output).strip()

    # Remove Markdown code fences if Qwen adds them
    raw_output = raw_output.replace("```json", "")
    raw_output = raw_output.replace("```", "")
    raw_output = raw_output.strip()

    try:
        parsed = json.loads(raw_output)

    except json.JSONDecodeError:

        start = raw_output.find("{")
        end = raw_output.rfind("}")

        if start == -1 or end == -1 or end < start:
            raise ValueError(
                "No JSON object found in model response."
            )

        parsed = json.loads(
            raw_output[start:end + 1]
        )

    if "diseases" not in parsed:
        raise ValueError(
            "JSON does not contain 'diseases'."
        )

    diseases = parsed["diseases"]

    if not isinstance(diseases, list):
        raise ValueError(
            "'diseases' must be a list."
        )

    cleaned = []
    seen = set()

    for disease in diseases:

        if not isinstance(disease, str):
            continue

        disease = disease.strip()

        if not disease:
            continue

        key = disease.lower()

        if key not in seen:
            seen.add(key)
            cleaned.append(disease)

    return cleaned

In [ ]:
def extract_qwen3(indication_text):

    user_message = EXTRACTION_TEMPLATE.format(
        indication_text=indication_text
    )

    payload = {
        "model": MODEL,

        "messages": [
            {
                "role": "system",
                "content": SYSTEM_MESSAGE
            },
            {
                "role": "user",
                "content": user_message
            }
        ],

        "stream": False,

        # Disable reasoning output
        "think": False,

        "options": {
            "temperature": 0.1,
            "num_ctx": 4096,
            "num_predict": 1000
        }
    }

    start_time = time.time()

    response = requests.post(
        URL,
        json=payload,
        timeout=240
    )

    response.raise_for_status()

    result = response.json()

    raw_output = result["message"]["content"]

    diseases = parse_output(raw_output)

    runtime = round(
        time.time() - start_time,
        2
    )

    return diseases, raw_output, runtime

In [ ]:
test_text = """
Metoprolol is indicated for the treatment of
hypertension and angina pectoris.
"""

diseases, raw, runtime = extract_qwen3(test_text)

print("RAW OUTPUT:")
print(raw)

print("\nPARSED OUTPUT:")
print(diseases)

print("\nRuntime:", runtime, "seconds")

RAW OUTPUT:
{"diseases":["hypertension","angina pectoris"]}

PARSED OUTPUT:
['hypertension', 'angina pectoris']

Runtime: 0.96 seconds


In [ ]:
test_text = """
Metoprolol is indicated for the treatment of
hypertension and angina pectoris.
"""

diseases, raw, runtime = extract_qwen3(test_text)

print("RAW OUTPUT:")
print(raw)

print("\nPARSED OUTPUT:")
print(diseases)

print("\nRuntime:", runtime, "seconds")

RAW OUTPUT:
{"diseases":["hypertension","angina pectoris"]}

PARSED OUTPUT:
['hypertension', 'angina pectoris']

Runtime: 0.64 seconds


In [ ]:
for i in range(5):

    record_id = df.loc[i, "record_id"]

    indication_text = str(
        df.loc[i, "indication_text"]
    )

    print()
    print("=" * 80)
    print(f"RECORD {i + 1}/5")
    print("Record ID:", record_id)

    print("\nGOLD:")
    print(
        df.loc[i, "gold_expert_mentions"]
    )

    try:

        diseases, raw, runtime = extract_qwen3(
            indication_text
        )

        df.at[
            i,
            "qwen3_output"
        ] = json.dumps(
            diseases,
            ensure_ascii=False
        )

        df.at[
            i,
            "qwen3_raw"
        ] = raw

        df.at[
            i,
            "qwen3_status"
        ] = (
            "OK"
            if diseases
            else "OK_EMPTY"
        )

        df.at[
            i,
            "qwen3_runtime_sec"
        ] = runtime

        print("\nQWEN OUTPUT:")
        print(diseases)

        print(
            "\nRuntime:",
            runtime,
            "seconds"
        )

    except Exception as e:

        df.at[
            i,
            "qwen3_status"
        ] = "FAILED"

        df.at[
            i,
            "qwen3_raw"
        ] = str(e)

        print("\nERROR:")
        print(e)


RECORD 1/5
Record ID: 10004

GOLD:
Erosive Esophagitis; Gastroesophageal Reflux Disease; GERD; Zollinger Ellison Syndrome

QWEN OUTPUT:
['Erosive Esophagitis Associated With Gastroesophageal Reflux Disease ( GERD )', 'Erosive Esophagitis', 'Maintenance of Healing of Erosive Esophagitis', 'Pathological Hypersecretory Conditions Including Zollinger Ellison Syndrome']

Runtime: 2.83 seconds

RECORD 2/5
Record ID: 10008

GOLD:
Anxiety Disorders; anxiety disorder; Anxiety; Anxiety associated with depression; Panic Disorder; agoraphobia

QWEN OUTPUT:
['anxiety disorder', 'Panic Disorder']

Runtime: 1.31 seconds

RECORD 3/5
Record ID: 10012

GOLD:
ventricular fibrillation; ventricular tachycardia; VT; VF

QWEN OUTPUT:
['ventricular fibrillation', 'hemodynamically unstable ventricular tachycardia', 'VT VF']

Runtime: 1.35 seconds

RECORD 4/5
Record ID: 10016

GOLD:
dermatoses; plaque type psoriasis

QWEN OUTPUT:
['inflammatory and pruritic manifestations of moderate to severe corticosteroid r

In [ ]:
display(
    df[
        [
            "record_id",
            "gold_expert_mentions",
            "qwen3_output",
            "qwen3_status",
            "qwen3_runtime_sec"
        ]
    ].head(5)
)

,record_id,gold_expert_mentions,qwen3_output,qwen3_status,qwen3_runtime_sec
0,10004,Erosive Esophagitis; Gastroesophageal Reflux D...,"[""Erosive Esophagitis Associated With Gastroes...",OK,2.83
1,10008,Anxiety Disorders; anxiety disorder; Anxiety; ...,"[""anxiety disorder"", ""Panic Disorder""]",OK,1.31
2,10012,ventricular fibrillation; ventricular tachycar...,"[""ventricular fibrillation"", ""hemodynamically ...",OK,1.35
3,10016,dermatoses; plaque type psoriasis,"[""inflammatory and pruritic manifestations of ...",OK,2.29
4,10017,"acute, painful","[""acute, painful musculoskeletal conditions""]",OK,0.65


In [ ]:
df.to_excel(
    RESULT_FILE,
    index=False
)

print("Saved:")
print(RESULT_FILE)

Saved:
/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_Qwen3_8B_Results.xlsx


In [ ]:
def run_all_qwen3():

    total = len(df)

    assert total == 500, (
        f"Expected 500 records but found {total}"
    )

    print("=" * 80)
    print("QWEN3 8B — FULL LABELEDIN RUN")
    print("Total records:", total)
    print("=" * 80)

    for i in range(total):

        record_id = df.loc[
            i,
            "record_id"
        ]

        current_status = str(
            df.at[
                i,
                "qwen3_status"
            ]
        ).strip()

        # Skip already completed records
        if current_status in [
            "OK",
            "OK_EMPTY",
            "OK_REPAIRED"
        ]:

            print(
                f"[{i + 1}/{total}] "
                f"Record {record_id} "
                "- already completed"
            )

            continue

        indication_text = str(
            df.loc[
                i,
                "indication_text"
            ]
        )

        print()
        print("=" * 80)

        print(
            f"[{i + 1}/{total}] "
            f"Record ID: {record_id}"
        )

        success = False
        last_error = ""

        # Try maximum 3 times
        for attempt in range(1, 4):

            try:

                diseases, raw, runtime = extract_qwen3(
                    indication_text
                )

                df.at[
                    i,
                    "qwen3_output"
                ] = json.dumps(
                    diseases,
                    ensure_ascii=False
                )

                df.at[
                    i,
                    "qwen3_raw"
                ] = raw

                df.at[
                    i,
                    "qwen3_status"
                ] = (
                    "OK"
                    if diseases
                    else "OK_EMPTY"
                )

                df.at[
                    i,
                    "qwen3_runtime_sec"
                ] = runtime

                print(
                    "Output:",
                    diseases
                )

                print(
                    "Runtime:",
                    runtime,
                    "seconds"
                )

                success = True
                break

            except Exception as e:

                last_error = str(e)

                print(
                    f"Attempt {attempt}/3 failed:",
                    e
                )

                time.sleep(2)

        if not success:

            df.at[
                i,
                "qwen3_status"
            ] = "FAILED"

            df.at[
                i,
                "qwen3_raw"
            ] = last_error

        # Save checkpoint every 20 records
        if (
            (i + 1) % 20 == 0
            or
            i == total - 1
        ):

            df.to_excel(
                RESULT_FILE,
                index=False
            )

            print(
                "\n>>> CHECKPOINT SAVED TO GOOGLE DRIVE"
            )

    # Final save
    df.to_excel(
        RESULT_FILE,
        index=False
    )

    print()
    print("=" * 80)
    print("QWEN3 8B FINISHED")
    print("=" * 80)

    print("Saved to:")
    print(RESULT_FILE)

In [ ]:
run_all_qwen3()

QWEN3 8B — FULL LABELEDIN RUN
Total records: 500
[1/500] Record 10004 - already completed
[2/500] Record 10008 - already completed
[3/500] Record 10012 - already completed
[4/500] Record 10016 - already completed
[5/500] Record 10017 - already completed

[6/500] Record ID: 1001
Output: ['Hypertension', 'Angina Pectoris', 'hemodynamically stable patients with definite or suspected acute myocardial infarction']
Runtime: 1.62 seconds

[7/500] Record ID: 10140
Output: ['major depressive disorder']
Runtime: 0.76 seconds

[8/500] Record ID: 10146
Output: ['insomnia']
Runtime: 0.51 seconds

[9/500] Record ID: 10153
Output: ['anxiety and tension associated with psychoneurosis', 'pruritus due to allergic conditions such as chronic urticaria and atopic and contact dermatoses', 'histamine mediated pruritus']
Runtime: 2.17 seconds

[10/500] Record ID: 10157
Output: ["Parkinson's disease", "idiopathic Parkinson's disease"]
Runtime: 0.92 seconds

[11/500] Record ID: 10173
Output: ['partial onset sei

In [ ]:
print(
    df["qwen3_status"]
    .value_counts(dropna=False)
)

qwen3_status
OK          499
OK_EMPTY      1
Name: count, dtype: int64


In [ ]:
completed = df[
    df["qwen3_status"].isin(
        [
            "OK",
            "OK_EMPTY",
            "OK_REPAIRED"
        ]
    )
]

failed = df[
    df["qwen3_status"] == "FAILED"
]

print("Total records :", len(df))
print("Completed     :", len(completed))
print("Failed        :", len(failed))

Total records : 500
Completed     : 500
Failed        : 0


In [ ]:
failed_rows = df[
    df["qwen3_status"] == "FAILED"
]

display(
    failed_rows[
        [
            "record_id",
            "gold_expert_mentions",
            "qwen3_raw"
        ]
    ]
)

,record_id,gold_expert_mentions,qwen3_raw


In [ ]:
failed_rows = df[
    df["qwen3_status"] == "FAILED"
]

display(
    failed_rows[
        [
            "record_id",
            "gold_expert_mentions",
            "qwen3_raw"
        ]
    ]
)

,record_id,gold_expert_mentions,qwen3_raw


In [ ]:
def parse_output_robust(raw_output):

    raw_output = str(raw_output).strip()

    raw_output = raw_output.replace("```json", "")
    raw_output = raw_output.replace("```", "")
    raw_output = raw_output.strip()

    decoder = json.JSONDecoder()

    try:

        parsed = json.loads(
            raw_output
        )

    except json.JSONDecodeError:

        start = raw_output.find("{")

        if start == -1:
            raise ValueError(
                "No JSON object found."
            )

        parsed, _ = decoder.raw_decode(
            raw_output[start:]
        )

    if "diseases" not in parsed:
        raise ValueError(
            "No diseases field."
        )

    diseases = parsed["diseases"]

    if not isinstance(diseases, list):
        raise ValueError(
            "'diseases' must be a list."
        )

    cleaned = []
    seen = set()

    for disease in diseases:

        if not isinstance(disease, str):
            continue

        disease = disease.strip()

        if not disease:
            continue

        key = disease.lower()

        if key not in seen:

            seen.add(key)
            cleaned.append(disease)

    return cleaned

In [ ]:
failed_indices = df.index[
    df["qwen3_status"] == "FAILED"
].tolist()

print(
    "Failed records to retry:",
    len(failed_indices)
)

for i in failed_indices:

    record_id = df.at[
        i,
        "record_id"
    ]

    indication_text = str(
        df.at[
            i,
            "indication_text"
        ]
    )

    print()
    print("=" * 80)
    print("Retrying:", record_id)

    try:

        user_message = EXTRACTION_TEMPLATE.format(
            indication_text=indication_text
        )

        payload = {
            "model": MODEL,

            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_MESSAGE
                },
                {
                    "role": "user",
                    "content": user_message
                }
            ],

            "stream": False,
            "think": False,

            "options": {
                "temperature": 0.1,
                "num_ctx": 4096,
                "num_predict": 1000
            }
        }

        start_time = time.time()

        response = requests.post(
            URL,
            json=payload,
            timeout=240
        )

        response.raise_for_status()

        raw = response.json()[
            "message"
        ][
            "content"
        ]

        runtime = round(
            time.time() - start_time,
            2
        )

        diseases = parse_output_robust(
            raw
        )

        df.at[
            i,
            "qwen3_output"
        ] = json.dumps(
            diseases,
            ensure_ascii=False
        )

        df.at[
            i,
            "qwen3_raw"
        ] = raw

        df.at[
            i,
            "qwen3_status"
        ] = (
            "OK"
            if diseases
            else "OK_EMPTY"
        )

        df.at[
            i,
            "qwen3_runtime_sec"
        ] = runtime

        print("Recovered:", diseases)

    except Exception as e:

        print(
            "Still failed:",
            e
        )

df.to_excel(
    RESULT_FILE,
    index=False
)

print("Retry results saved.")

Failed records to retry: 0
Retry results saved.


In [ ]:
df.to_excel(
    RESULT_FILE,
    index=False
)

print("=" * 80)
print("FINAL QWEN FILE SAVED")
print("=" * 80)

print(
    RESULT_FILE
)

FINAL QWEN FILE SAVED
/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_Qwen3_8B_Results.xlsx
